# LangChain: Q&A over Documents

Load a CSV product catalog, embed it into an in-memory vector store,
and answer natural-language questions about it using retrieval-augmented generation (RAG).

**Concept flow:**
```
CSV file
   │  CSVLoader
   ▼
Documents  ──  OpenAIEmbeddings  ──►  DocArrayInMemorySearch (vector store)
                                              │
                                         retriever
                                              │
                              question ──► RAG chain ──► answer
```

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

llm_model = 'gpt-3.5-turbo'

In [ ]:
# Install if missing:
# pip install langchain-openai langchain-community docarray

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from IPython.display import display, Markdown

## Step 1 — Load documents from CSV

`CSVLoader` reads each row as a separate `Document`.
Each document's `page_content` contains the row data as key: value text.

In [ ]:
# Place OutdoorClothingCatalog_1000.csv in the assets/ folder
loader = CSVLoader(file_path='assets/OutdoorClothingCatalog_1000.csv')
docs = loader.load()

print(f'Loaded {len(docs)} documents')
print('\nFirst document:')
print(docs[0].page_content[:300])

## Step 2 — Embed documents into a vector store

`OpenAIEmbeddings` converts text into a 1536-dimensional numeric vector.
Similar text produces similar vectors — this is what makes semantic search possible.

`DocArrayInMemorySearch` stores those vectors in RAM (no external DB needed).

In [ ]:
embeddings = OpenAIEmbeddings()

# Quick sanity check — embed a single sentence and inspect the vector
sample_vector = embeddings.embed_query('Hi my name is Harrison')
print(f'Embedding dimensions : {len(sample_vector)}')
print(f'First 5 values       : {sample_vector[:5]}')

In [ ]:
# Build the vector store from all loaded documents
vectorstore = DocArrayInMemorySearch.from_documents(docs, embeddings)

## Step 3 — Similarity search

A `retriever` wraps the vector store and finds the `k` most relevant documents
for any given query using cosine similarity between embeddings.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

# Test retrieval directly
query = 'Please suggest a shirt with sunblocking'
relevant_docs = retriever.invoke(query)

print(f'Retrieved {len(relevant_docs)} documents')
print('\nTop result:')
print(relevant_docs[0].page_content)

## Step 4 — Build the RAG chain

RAG = **R**etrieval-**A**ugmented **G**eneration.

Instead of asking the LLM from memory, we:
1. Retrieve the relevant documents for the question
2. Inject them as `context` into the prompt
3. Let the LLM answer based only on that context

This is the modern LCEL replacement for the deprecated `RetrievalQA` chain.

In [ ]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

prompt = ChatPromptTemplate.from_template(
    'Answer the question based only on the following context:\n\n'
    '{context}\n\n'
    'Question: {question}'
)

def format_docs(documents):
    """Join retrieved document texts into a single context string."""
    return '\n\n'.join(doc.page_content for doc in documents)

# LCEL pipeline:
#   question  ──►  retriever  ──►  format_docs  ──►  prompt  ──►  llm  ──►  string
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## Step 5 — Ask questions

In [ ]:
question = (
    'Please list all your shirts with sun protection '
    'in a table in markdown and summarize each one.'
)

response = rag_chain.invoke(question)
display(Markdown(response))

In [ ]:
response = rag_chain.invoke('Do you have any waterproof jackets?')
display(Markdown(response))

## Bonus — One-liner shortcut with `VectorstoreIndexCreator`

`VectorstoreIndexCreator` wraps all of Steps 1–3 into a single call.
Useful for quick experiments; the step-by-step approach above gives more control.

In [ ]:
from langchain_community.indexes import VectorstoreIndexCreator

index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=OpenAIEmbeddings(),
).from_loaders([CSVLoader(file_path='assets/OutdoorClothingCatalog_1000.csv')])

response = index.query(question, llm=ChatOpenAI(temperature=0, model=llm_model))
display(Markdown(response))